# Evaluate and save a meeting summary

Use native DeepEval alignment and coverage, with transcript-specific coverage questions
saved for reuse. The first question-preparation call generates questions; later calls load
the same set. Review them before evaluating a summary. There is no format grade or
acceptance gate.

Install with `python -m pip install -e '.[notebooks]'` in the repository environment and
select that environment as this notebook's kernel. Set `OPENAI_API_KEY` in your environment
or the repository's local `.env`. Optional `OPENAI_MODEL` and `OPENAI_BASE_URL` select the
judge and endpoint. **Preparing new questions and evaluating summaries send source text
to that endpoint and make billable model calls. Saving and exporting make no model calls.**
All bundled examples are fictional.


In [ ]:
from pathlib import Path
import os

# Work from either the repository root or its notebooks directory.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "examples" / "data").is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
os.chdir(root)

from meeting_eval import (
    create_run_directory,
    evaluate_meeting,
    get_or_prepare_questions,
    new_run,
    save_evaluation,
)


## Choose the meeting and load sources

The transcript is the sole factual source. Business context describes audience and purpose
and influences question selection only. Set `business_context = None` to omit it.
Change `meeting_id` or these file paths to inspect your own meeting.

Leave `n = None` to generate five questions for a new meeting or reuse the saved count.
Set a positive integer when preparing a different sample size. Changed transcript,
business context, or requested question count requires explicit regeneration. Set
`regenerate_questions = True` only when deliberately replacing the saved question set,
then return it to `False`. Earlier saved evaluations retain their exact question sets.


In [ ]:
meeting_id = "meeting_001"
transcript_path = Path(f"examples/data/transcripts/{meeting_id}.txt")
summary_path = Path(f"examples/data/summaries/{meeting_id}.md")
transcript = transcript_path.read_text(encoding="utf-8-sig")
summary = summary_path.read_text(encoding="utf-8-sig")
business_context = Path("examples/business-context.md").read_text(encoding="utf-8-sig")

model = None
n = None
regenerate_questions = False
questions_dir = Path("results/questions")
output_dir = Path("results")

if not transcript.strip() or not summary.strip():
    raise ValueError("Transcript and summary must both contain text.")


## Prepare or load the frozen questions

Run this cell to see the exact factual questions before spending evaluation calls.
On first use, it generates and saves the questions; otherwise it loads the saved set.
These are transcript-specific native coverage questions. The historical
[content checklist](../docs/fixed-evaluation-questions.md) guides topic selection and
receives no separate rubric scores.

Reusing questions removes question-selection variation. Claim extraction and judge
answers can still vary between evaluations.


In [ ]:
question_set, questions_generated = await get_or_prepare_questions(
    meeting_id,
    transcript,
    business_context=business_context,
    questions_dir=questions_dir,
    model=model,
    n=n,
    regenerate=regenerate_questions,
)
print("Generated and saved questions." if questions_generated else "Loaded saved questions.")
for index, question in enumerate(question_set["questions"], start=1):
    print(f"Q{index:02}: {question}")
print("\nQuestion-set provenance:")
for key, value in question_set.items():
    if key != "questions":
        print(f"{key}: {value}")


## Evaluate once and save immediately

This cell evaluates the summary against the loaded question set, using a fresh native
metric, and saves the complete result before any report rendering. Each execution creates
a new run directory. The saved JSON is the source for both Excel and HTML exports.


In [ ]:
run_dir = create_run_directory(output_dir)
run = new_run()
result = await evaluate_meeting(
    meeting_id,
    transcript,
    summary,
    question_set=question_set,
    model=model,
    transcript_path=transcript_path,
    summary_path=summary_path,
    question_generation_performed=questions_generated,
)
run["results"] = [result]
run["status"] = "completed"
results_json = save_evaluation(run, run_dir / "results.json").resolve()
questions_generated = False
print("Saved evaluation:", results_json)


## Inspect the saved evaluation's details

Alignment is the fraction of extracted summary claims judged supported. Coverage is the
fraction of source-answerable frozen questions also answerable from the summary.
Coverage is a sample, not the percentage of all important information retained. Overall is
`min(alignment, coverage)`.

Read counts and verdicts alongside scores. Derived outreach ideas absent from the
transcript may lower native alignment even when labeled as suggestions. Model extraction
and judging can miss material errors; the transcript establishes what was said, not
externally verified truth.


In [ ]:
print("Execution:", result["status"])
print("Scores:", result["scores"])
print("Counts:", result["counts"])
print("Reason:", result.get("reason"))
if result.get("error"):
    print("Error:", result["error"])

print("\nClaims and alignment verdicts:")
for item in result["alignment"]:
    print(item["claim_id"], item["claim"])
    print(item["verdict"], item.get("reason"))

print("\nCoverage verdicts:")
for item in result["coverage"]:
    print(item["question_id"], item["question"])
    print("Transcript:", item["transcript_answer"], "Summary:", item["summary_answer"])
    print("Eligible:", item["in_denominator"], "Outcome:", item["outcome"])


## Export the saved result to Excel and HTML

This cell reads `results_json` and creates `report.xlsx` and `report.html` beside it.
It does not prepare questions, evaluate again, or reload the source files. Both reports
contain the same recorded scores, counts, questions, claims, and verdicts.

To export after restarting the kernel, run only this cell after uncommenting its
`results_json = Path(...)` line and replacing the path with a previously saved JSON file.
No source-loading or evaluation cell is needed.


In [ ]:
from pathlib import Path
from meeting_eval import export_reports

# After a kernel restart, choose an existing saved evaluation:
# results_json = Path("/absolute/path/to/results/<run-directory>/results.json")
report_dir = export_reports(results_json)
print("Excel:", report_dir / "report.xlsx")
print("HTML:", report_dir / "report.html")


## Evaluate folders or re-export from the command line

Use the following commands from the repository root. Batch evaluation pairs flat
`.txt`/`.md` folders, automatically prepares missing question sets, and reuses existing
sets. The report command reads only saved JSON and makes no model calls.

```bash
.venv/bin/meeting-eval evaluate \
  --transcripts examples/data/transcripts \
  --summaries examples/data/summaries \
  --business-context examples/business-context.md \
  --questions results/questions \
  --output results

.venv/bin/meeting-eval report results/<run-directory>/results.json
```

For offline checks, run `.venv/bin/python -m pytest`. See the [README](../README.md) for
setup, outputs, and limitations. No human calibration or comparative rubric scoring is
performed by this workflow.
